# 🧬 Train DenseNet-121 on Breast Histopathology (IDC)

**Med-Image CompareNet — Module 2 (CNN Classification)**

This notebook trains DenseNet-121 on the **Breast Histopathology Images** dataset
(Invasive Ductal Carcinoma detection: Benign vs Malignant).

| Setting | Value |
|---------|-------|
| Model | DenseNet-121 (ImageNet pretrained) |
| Dataset | Breast Histopathology IDC |
| Classes | Benign (0) / Malignant (1) |
| Image Size | 224×224 |
| Epochs | 30 |
| Batch Size | 32 |
| LR | 0.0001 |
| Optimizer | Adam |
| Early Stopping | Patience 7 |

**Output:** `densenet121_pathology_best.pth` → download and place in `checkpoints/`

## 1. Setup & GPU Check

In [ ]:
import torch
print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB")
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using: {device}")

## 2. Install Dependencies

In [ ]:
!pip install -q scikit-learn tqdm

## 3. Download Breast Histopathology Dataset from Kaggle

In [ ]:
# Upload your Kaggle API key (kaggle.json) first
# Go to kaggle.com -> Account -> Create New API Token -> upload the kaggle.json
import os

# Option 1: Upload kaggle.json manually
from google.colab import files
print("Upload your kaggle.json file:")
uploaded = files.upload()

!mkdir -p ~/.kaggle
!mv kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json
print("Kaggle API configured!")

In [ ]:
# Download the Breast Histopathology Images dataset
!pip install -q kaggle
!kaggle datasets download -d paultimothymooney/breast-histopathology-images -p /content/data/ --unzip
print("\nDataset downloaded!")

In [ ]:
# Verify dataset structure
import os

# The dataset may extract into different structures - let's find the patient directories
data_root = '/content/data'
print("Top-level contents:")
for item in sorted(os.listdir(data_root))[:20]:
    full = os.path.join(data_root, item)
    if os.path.isdir(full):
        print(f"  📁 {item}/  ({len(os.listdir(full))} items)")
    else:
        print(f"  📄 {item}")

# Find the actual root with patient directories (they are named with IDs like 8863, 8864, ...)
# The dataset structure is: root/<patient_id>/{0,1}/*.png
# Sometimes it extracts inside a subfolder
candidate_roots = [data_root]
for item in os.listdir(data_root):
    full = os.path.join(data_root, item)
    if os.path.isdir(full):
        candidate_roots.append(full)

DATASET_ROOT = None
for root in candidate_roots:
    # Check if this contains patient directories with 0/ and 1/ subdirs
    subdirs = [d for d in os.listdir(root) if os.path.isdir(os.path.join(root, d))]
    for sd in subdirs[:5]:
        sub_path = os.path.join(root, sd)
        sub_items = os.listdir(sub_path) if os.path.isdir(sub_path) else []
        if '0' in sub_items or '1' in sub_items:
            DATASET_ROOT = root
            break
    if DATASET_ROOT:
        break

if DATASET_ROOT:
    patient_dirs = [d for d in os.listdir(DATASET_ROOT) if os.path.isdir(os.path.join(DATASET_ROOT, d))]
    print(f"\n✅ Found dataset root: {DATASET_ROOT}")
    print(f"   Number of patient directories: {len(patient_dirs)}")
else:
    print("❌ Could not auto-detect dataset root. Check the directory structure above.")
    DATASET_ROOT = data_root  # fallback

## 4. Dataset & DataLoader Setup

Matches the project's `src/datasets.py` exactly:
- Max 20,000 patches per class
- Stratified 70/15/15 train/val/test split
- Weighted random sampling for class imbalance
- ImageNet normalization

In [ ]:
import os
import numpy as np
from pathlib import Path
from typing import List, Tuple, Optional, Dict
from PIL import Image
from sklearn.model_selection import train_test_split

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from torchvision import transforms, models
from tqdm import tqdm
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, classification_report, confusion_matrix,
)
import copy, time, json

# ─── Reproducibility ───
SEED = 42
import random
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False
os.environ['PYTHONHASHSEED'] = str(SEED)
print(f"Seed set to {SEED}")

In [ ]:
# ─── Transforms (matches project's get_transforms) ───
IMAGE_SIZE = 224
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]

train_transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE + 32, IMAGE_SIZE + 32)),
    transforms.RandomCrop(IMAGE_SIZE),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(degrees=15),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.1, hue=0.05),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
    transforms.RandomErasing(p=0.1),
])

eval_transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

print("Transforms ready.")

In [ ]:
# ─── Dataset class (matches project's BreastHistopathologyDataset) ───
class BreastHistopathologyDataset(Dataset):
    def __init__(self, root_dir, split='train', transform=None, max_per_class=20000, seed=42):
        self.root_dir = Path(root_dir)
        self.transform = transform
        self.classes = ['benign', 'malignant']

        # Collect all image paths
        all_paths_0 = []  # benign
        all_paths_1 = []  # malignant

        for patient_dir in self.root_dir.iterdir():
            if not patient_dir.is_dir():
                continue
            benign_dir = patient_dir / '0'
            malignant_dir = patient_dir / '1'

            if benign_dir.exists():
                all_paths_0.extend([str(p) for p in benign_dir.glob('*.png')])
            if malignant_dir.exists():
                all_paths_1.extend([str(p) for p in malignant_dir.glob('*.png')])

        print(f"Found {len(all_paths_0)} benign + {len(all_paths_1)} malignant patches")

        # Cap per class
        rng = np.random.RandomState(seed)
        if len(all_paths_0) > max_per_class:
            all_paths_0 = list(rng.choice(all_paths_0, max_per_class, replace=False))
        if len(all_paths_1) > max_per_class:
            all_paths_1 = list(rng.choice(all_paths_1, max_per_class, replace=False))

        all_paths = all_paths_0 + all_paths_1
        all_labels = [0] * len(all_paths_0) + [1] * len(all_paths_1)

        # Stratified train/val/test split (70/15/15)
        train_paths, temp_paths, train_labels, temp_labels = train_test_split(
            all_paths, all_labels, test_size=0.3, stratify=all_labels, random_state=seed
        )
        val_paths, test_paths, val_labels, test_labels = train_test_split(
            temp_paths, temp_labels, test_size=0.5, stratify=temp_labels, random_state=seed
        )

        split_map = {
            'train': (train_paths, train_labels),
            'val': (val_paths, val_labels),
            'test': (test_paths, test_labels),
        }

        self.image_paths, self.labels = split_map[split]
        print(f"  [{split}]: {len(self)} patches "
              f"({sum(1 for l in self.labels if l == 0)} benign, "
              f"{sum(1 for l in self.labels if l == 1)} malignant)")

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        img = Image.open(self.image_paths[idx]).convert('RGB')
        label = self.labels[idx]
        if self.transform:
            img = self.transform(img)
        return img, label

    def get_class_weights(self):
        counts = np.bincount(self.labels)
        weights = 1.0 / counts
        weights = weights / weights.sum()
        return torch.tensor(weights, dtype=torch.float32)

In [ ]:
# ─── Create datasets and dataloaders ───
MAX_PER_CLASS = 20000
BATCH_SIZE = 32
NUM_WORKERS = 4

print("Loading datasets...")
train_ds = BreastHistopathologyDataset(DATASET_ROOT, 'train', train_transform, MAX_PER_CLASS, SEED)
val_ds   = BreastHistopathologyDataset(DATASET_ROOT, 'val',   eval_transform,  MAX_PER_CLASS, SEED)
test_ds  = BreastHistopathologyDataset(DATASET_ROOT, 'test',  eval_transform,  MAX_PER_CLASS, SEED)

# Weighted sampler for class imbalance
train_weights = train_ds.get_class_weights()
sample_weights = [train_weights[l].item() for l in train_ds.labels]
sampler = WeightedRandomSampler(sample_weights, len(sample_weights), replacement=True)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, sampler=sampler, num_workers=NUM_WORKERS, pin_memory=True, drop_last=True)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False,   num_workers=NUM_WORKERS, pin_memory=True)
test_loader  = DataLoader(test_ds,  batch_size=BATCH_SIZE, shuffle=False,   num_workers=NUM_WORKERS, pin_memory=True)

print(f"\n✅ DataLoaders ready:")
print(f"   Train: {len(train_ds)} samples, {len(train_loader)} batches")
print(f"   Val:   {len(val_ds)} samples, {len(val_loader)} batches")
print(f"   Test:  {len(test_ds)} samples, {len(test_loader)} batches")

## 5. Build DenseNet-121 Model

Same architecture as the project's `build_cnn_model('densenet121', 2)`

In [ ]:
# Build DenseNet-121 with ImageNet pretrained weights
def build_densenet121(num_classes=2, pretrained=True):
    weights = models.DenseNet121_Weights.IMAGENET1K_V1 if pretrained else None
    model = models.densenet121(weights=weights)
    model.classifier = nn.Sequential(
        nn.Dropout(0.3),
        nn.Linear(model.classifier.in_features, num_classes),
    )
    return model

model = build_densenet121(num_classes=2, pretrained=True).to(device)

# Count parameters
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"DenseNet-121 loaded on {device}")
print(f"  Total params: {total_params:,}")
print(f"  Trainable:    {trainable_params:,}")

## 6. Training Configuration

Matches `config.yaml` CNN training settings exactly.

In [ ]:
# ─── Hyperparameters (from config.yaml) ───
EPOCHS = 30
LEARNING_RATE = 0.0001
WEIGHT_DECAY = 0.0001
LR_STEP_SIZE = 10
LR_GAMMA = 0.1
PATIENCE = 7
USE_AMP = True  # Mixed precision for speed on T4

optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=LR_STEP_SIZE, gamma=LR_GAMMA)
criterion = nn.CrossEntropyLoss()
scaler = torch.amp.GradScaler() if USE_AMP else None

print("Training config:")
print(f"  Epochs: {EPOCHS}")
print(f"  LR: {LEARNING_RATE}")
print(f"  Weight Decay: {WEIGHT_DECAY}")
print(f"  LR Schedule: StepLR(step={LR_STEP_SIZE}, gamma={LR_GAMMA})")
print(f"  Early Stopping Patience: {PATIENCE}")
print(f"  Mixed Precision: {USE_AMP}")

## 7. Training Loop

In [ ]:
# ─── Training Loop ───
history = {'train_loss': [], 'val_loss': [], 'train_acc': [], 'val_acc': []}
best_val_acc = 0.0
patience_counter = 0
best_weights = None

print(f"\n{'='*70}")
print(f"  Training DenseNet-121 on Breast Histopathology (IDC)")
print(f"{'='*70}\n")

for epoch in range(1, EPOCHS + 1):
    # ── Train ──
    model.train()
    running_loss, correct, total = 0.0, 0, 0
    pbar = tqdm(train_loader, desc=f'Epoch {epoch}/{EPOCHS} [Train]', leave=False)
    
    for images, labels in pbar:
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        
        if USE_AMP:
            with torch.amp.autocast(device_type='cuda'):
                outputs = model(images)
                loss = criterion(outputs, labels)
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
        else:
            outputs = model(images)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
        
        running_loss += loss.item() * images.size(0)
        _, preds = outputs.max(1)
        correct += preds.eq(labels).sum().item()
        total += labels.size(0)
        pbar.set_postfix({'loss': f'{loss.item():.4f}', 'acc': f'{correct/total:.4f}'})
    
    scheduler.step()
    train_loss = running_loss / total
    train_acc = correct / total
    
    # ── Validate ──
    model.eval()
    val_loss, val_correct, val_total = 0.0, 0, 0
    with torch.no_grad():
        for images, labels in val_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            loss = criterion(outputs, labels)
            val_loss += loss.item() * images.size(0)
            _, preds = outputs.max(1)
            val_correct += preds.eq(labels).sum().item()
            val_total += labels.size(0)
    
    val_loss = val_loss / val_total
    val_acc = val_correct / val_total
    
    history['train_loss'].append(train_loss)
    history['val_loss'].append(val_loss)
    history['train_acc'].append(train_acc)
    history['val_acc'].append(val_acc)
    
    print(f'Epoch {epoch:3d}/{EPOCHS} │ '
          f'Train Loss: {train_loss:.4f} Acc: {train_acc:.4f} │ '
          f'Val Loss: {val_loss:.4f} Acc: {val_acc:.4f} │ '
          f'LR: {scheduler.get_last_lr()[0]:.6f} │ '
          f'Patience: {patience_counter}/{PATIENCE}')
    
    # ── Early Stopping ──
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        best_weights = copy.deepcopy(model.state_dict())
        patience_counter = 0
        
        # Save checkpoint (same format as project's save_checkpoint)
        torch.save({
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'epoch': epoch,
            'metrics': {'accuracy': val_acc, 'loss': val_loss},
        }, 'densenet121_pathology_best.pth')
        print(f'  💾 New best model saved! Val Acc: {val_acc:.4f}')
    else:
        patience_counter += 1
        if patience_counter >= PATIENCE:
            print(f'\n⏹️  Early stopping triggered at epoch {epoch}')
            break

# Restore best weights
if best_weights:
    model.load_state_dict(best_weights)
print(f'\n✅ Training complete! Best Val Accuracy: {best_val_acc:.4f}')

## 8. Plot Training Curves

In [ ]:
import matplotlib.pyplot as plt

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

epochs_range = range(1, len(history['train_loss']) + 1)

# Loss
ax1.plot(epochs_range, history['train_loss'], 'b-', label='Train Loss', linewidth=2)
ax1.plot(epochs_range, history['val_loss'], 'r-', label='Val Loss', linewidth=2)
ax1.set_xlabel('Epoch')
ax1.set_ylabel('Loss')
ax1.set_title('DenseNet-121 Pathology — Loss')
ax1.legend()
ax1.grid(True, alpha=0.3)

# Accuracy
ax2.plot(epochs_range, history['train_acc'], 'b-', label='Train Acc', linewidth=2)
ax2.plot(epochs_range, history['val_acc'], 'r-', label='Val Acc', linewidth=2)
ax2.set_xlabel('Epoch')
ax2.set_ylabel('Accuracy')
ax2.set_title('DenseNet-121 Pathology — Accuracy')
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('densenet121_pathology_training_curves.png', dpi=150, bbox_inches='tight')
plt.show()
print('📊 Training curves saved!')

## 9. Full Evaluation on Test Set

In [ ]:
# ─── Evaluate on test set ───
model.eval()
all_preds, all_labels, all_probs = [], [], []
total_time = 0.0

with torch.no_grad():
    for images, labels in tqdm(test_loader, desc='Testing'):
        images = images.to(device)
        start = time.time()
        outputs = model(images)
        total_time += time.time() - start
        probs = torch.softmax(outputs, dim=1)
        _, preds = outputs.max(1)
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.numpy())
        all_probs.extend(probs[:, 1].cpu().numpy())

y_true = np.array(all_labels)
y_pred = np.array(all_preds)
y_prob = np.array(all_probs)
n = len(y_true)

results = {
    'accuracy': accuracy_score(y_true, y_pred),
    'precision': precision_score(y_true, y_pred, average='binary', zero_division=0),
    'recall': recall_score(y_true, y_pred, average='binary', zero_division=0),
    'f1': f1_score(y_true, y_pred, average='binary', zero_division=0),
    'auc_roc': roc_auc_score(y_true, y_prob) if len(set(y_true)) > 1 else 0.0,
    'inference_time_ms': (total_time / n) * 1000,
    'total_samples': n,
}

print(f"\n{'='*60}")
print(f"  DenseNet-121 Pathology — Test Results")
print(f"{'='*60}")
print(f"  Accuracy:    {results['accuracy']:.4f}")
print(f"  Precision:   {results['precision']:.4f}")
print(f"  Recall:      {results['recall']:.4f}")
print(f"  F1-Score:    {results['f1']:.4f}")
print(f"  AUC-ROC:     {results['auc_roc']:.4f}")
print(f"  Inference:   {results['inference_time_ms']:.2f} ms/image")
print(f"  Samples:     {results['total_samples']}")
print(f"{'='*60}")

# Confusion Matrix
cm = confusion_matrix(y_true, y_pred)
print(f"\nConfusion Matrix:")
print(f"  {'':>12} Pred Benign  Pred Malignant")
print(f"  {'True Benign':>12}     {cm[0][0]:>5}         {cm[0][1]:>5}")
print(f"  {'True Malig.':>12}     {cm[1][0]:>5}         {cm[1][1]:>5}")

# Classification Report
print(f"\n{classification_report(y_true, y_pred, target_names=['Benign', 'Malignant'])}")

## 10. Save Results & Download Checkpoint

In [ ]:
# Save results JSON (for dashboard)
with open('densenet121_pathology_results.json', 'w') as f:
    json.dump(results, f, indent=2)
print('📄 Results saved to densenet121_pathology_results.json')

# Verify checkpoint file
ckpt_size = os.path.getsize('densenet121_pathology_best.pth') / (1024*1024)
print(f'\n💾 Checkpoint: densenet121_pathology_best.pth ({ckpt_size:.1f} MB)')

# Verify it can be loaded
ckpt = torch.load('densenet121_pathology_best.pth', map_location='cpu')
print(f'   Epoch: {ckpt["epoch"]}')
print(f'   Metrics: {ckpt["metrics"]}')
print(f'   Keys: {list(ckpt.keys())}')

In [ ]:
# ─── Download the checkpoint ───
from google.colab import files
print("Downloading densenet121_pathology_best.pth...")
print("Place this file in your project's checkpoints/ folder.")
files.download('densenet121_pathology_best.pth')

In [ ]:
# Also download the results JSON and training curves
files.download('densenet121_pathology_results.json')
files.download('densenet121_pathology_training_curves.png')
print("\n✅ All files downloaded!")
print("\nNext steps:")
print("  1. Move densenet121_pathology_best.pth → checkpoints/")
print("  2. Restart the Streamlit dashboard")
print("  3. Select DenseNet-121 + Pathology (IDC) in the sidebar")
print("  4. All 6 model-dataset combinations are now complete! 🎉")